# Day 11 · Delta Lake Advanced
## Changing what is already there

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

Last session ended on two open questions.

| | |
|---|---|
| You can add rows and remove rows | but you cannot **change** one |
| Every version is kept | and nobody said what that **costs** |

Tonight answers both.

## Configuration

In [ ]:
from pyspark.sql import functions as F

CATALOG      = "workspace"
SCHEMA       = "day11"
SOURCE_TABLE = "workspace.default.upi_transactions_2026"

DIM   = f"{CATALOG}.{SCHEMA}.customers"
SCD   = f"{CATALOG}.{SCHEMA}.customers_history"
BUSY  = f"{CATALOG}.{SCHEMA}.txn_many_files"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
for _t in (DIM, SCD, BUSY):
    spark.sql(f"DROP TABLE IF EXISTS {_t}")

print(f"dimension  : {DIM}")
print(f"history    : {SCD}")
print(f"many files : {BUSY}")

---
# 1 · The missing verb

**Industry example.** A customer moves from Pune to Bengaluru. One row, one column, one
new value. On a plain data lake that sentence is a rewrite job.

Eight customers. Small enough to read every row on screen.

In [ ]:
customers = spark.createDataFrame(
    [(101, "Aarti Rao",     "Hyderabad", "PREMIER"),
     (102, "Vikram Shah",   "Mumbai",    "PLUS"),
     (103, "Neha Iyer",     "Bengaluru", "PREMIER"),
     (104, "Rahul Menon",   "Pune",      "BASIC"),
     (105, "Sana Qureshi",  "Delhi",     "PLUS"),
     (106, "Joseph Thomas", "Chennai",   "BASIC"),
     (107, "Priya Nair",    "Kochi",     "PREMIER"),
     (108, "Amit Bansal",   "Jaipur",    "PLUS")],
    "customer_id INT, name STRING, city STRING, tier STRING",
)

customers.write.format("delta").mode("overwrite").saveAsTable(DIM)

display(spark.table(DIM).orderBy("customer_id"))

### One row, one column

In [ ]:
spark.sql(f"UPDATE {DIM} SET city = 'Bengaluru' WHERE customer_id = 104")

display(spark.sql(f"SELECT * FROM {DIM} WHERE customer_id = 104"))

In [ ]:
spark.sql(f"DELETE FROM {DIM} WHERE tier = 'BASIC'")

display(spark.table(DIM).orderBy("customer_id"))

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {DIM}")
         .select("version", "operation", "operationMetrics")
         .orderBy("version")
)

### What actually happened

Parquet files are immutable — nothing was edited in place. To change one row, Delta read
the file containing it, wrote a **new** file with the row changed, and recorded in the log
that the old file no longer counts.

Read `numTargetFilesAdded` and `numTargetFilesRemoved` above. That is the cost of a single
-row update, and it is why Delta is built for analytics rather than for a booking system
handling thousands of small writes a second.

---
# 2 · MERGE — one statement for a mixed file

**Industry example.** Every night the CRM sends one file. Some rows are customers you have
never seen. Some are customers whose details changed. The file does not tell you which is
which.

Without `MERGE` you would work that out yourself: join to find the overlap, split the batch
in two, update one part, insert the other, and hope nothing failed between the two writes.

In [ ]:
todays_feed = spark.createDataFrame(
    [(101, "Aarti Rao",    "Hyderabad", "ELITE"),
     (102, "Vikram Shah",  "Pune",      "PLUS"),
     (109, "Farah Sheikh", "Lucknow",   "PREMIER"),
     (110, "Deepak Verma", "Indore",    "BASIC")],
    "customer_id INT, name STRING, city STRING, tier STRING",
)

todays_feed.createOrReplaceTempView("todays_feed")

display(spark.table("todays_feed"))

### One statement, both outcomes

In [ ]:
spark.sql(f"""
    MERGE INTO {DIM} AS target
    USING todays_feed AS source
      ON target.customer_id = source.customer_id
    WHEN MATCHED THEN UPDATE SET
        target.name = source.name,
        target.city = source.city,
        target.tier = source.tier
    WHEN NOT MATCHED THEN INSERT
        (customer_id, name, city, tier)
        VALUES (source.customer_id, source.name, source.city, source.tier)
""")

display(spark.table(DIM).orderBy("customer_id"))

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {DIM}")
         .select("version", "operation", "operationMetrics")
         .filter("operation = 'MERGE'")
)

### The metrics are the lesson

`numTargetRowsUpdated` and `numTargetRowsInserted` say exactly how the file split. You never
had to know in advance, and the whole thing was **one commit** — so it either all happened
or none of it did.

---
# 3 · Slowly Changing Dimensions, Type 2

The merge above destroyed something. Vikram Shah used to live in Mumbai; now the table says
Pune and there is no trace of Mumbai.

| | |
|---|---|
| **Type 1** | overwrite the value. Simple, and the past is gone |
| **Type 2** | close the old row, open a new one. The past is a row of its own |

**Industry example.** Revenue by city for last quarter. If a customer moved in March, last
quarter's revenue belongs to the old city — and a Type 1 dimension will silently give it to
the new one.

In [ ]:
(spark.table(DIM)
   .withColumn("valid_from", F.lit("2026-01-01").cast("date"))
   .withColumn("valid_to",   F.lit(None).cast("date"))
   .withColumn("is_current", F.lit(True))
   .write.format("delta").mode("overwrite").saveAsTable(SCD))

display(spark.table(SCD).orderBy("customer_id"))

### Why one MERGE is not enough

A changed customer needs **two** things to happen: the current row must be closed, and a new
row must be opened. A single `MERGE` gives each source row one outcome — matched or not
matched — so it cannot do both to the same key.

The standard trick is to feed `MERGE` **two rows per change**: one that matches the existing
record and closes it, and one with a deliberately `NULL` key so that it cannot match anything
and is therefore inserted.

In [ ]:
changes = spark.createDataFrame(
    [(102, "Vikram Shah", "Bengaluru", "PREMIER"),
     (103, "Neha Iyer",   "Bengaluru", "ELITE"),
     (111, "Ravi Kumar",  "Nagpur",    "BASIC")],
    "customer_id INT, name STRING, city STRING, tier STRING",
)
changes.createOrReplaceTempView("changes")

display(spark.sql(f"""
    SELECT c.customer_id AS merge_key, c.*
    FROM changes c
    UNION ALL
    SELECT NULL AS merge_key, c.*
    FROM changes c
    JOIN {SCD} h
      ON h.customer_id = c.customer_id
     AND h.is_current  = true
     AND (h.city <> c.city OR h.tier <> c.tier)
    ORDER BY customer_id, merge_key NULLS LAST
"""))

### The staged merge

In [ ]:
spark.sql(f"""
    MERGE INTO {SCD} AS h
    USING (
        SELECT c.customer_id AS merge_key, c.*
        FROM changes c
        UNION ALL
        SELECT NULL AS merge_key, c.*
        FROM changes c
        JOIN {SCD} h2
          ON h2.customer_id = c.customer_id
         AND h2.is_current  = true
         AND (h2.city <> c.city OR h2.tier <> c.tier)
    ) AS staged
      ON h.customer_id = staged.merge_key AND h.is_current = true
    WHEN MATCHED AND (h.city <> staged.city OR h.tier <> staged.tier) THEN UPDATE SET
        h.is_current = false,
        h.valid_to   = current_date()
    WHEN NOT MATCHED THEN INSERT
        (customer_id, name, city, tier, valid_from, valid_to, is_current)
        VALUES (staged.customer_id, staged.name, staged.city, staged.tier,
                current_date(), NULL, true)
""")

display(spark.sql(f"""
    SELECT * FROM {SCD}
    WHERE customer_id IN (102, 103, 111)
    ORDER BY customer_id, valid_from
"""))

### Reading it back

Two rows for a customer who moved. One closed, one open. The dimension can now answer
"where did this customer live in March" as well as "where do they live now".

In [ ]:
display(spark.sql(f"SELECT * FROM {SCD} WHERE is_current = true ORDER BY customer_id"))

---
# 4 · Many small files

**Industry example.** A job appends every fifteen minutes. After a month the table holds a
reasonable amount of data spread across a few thousand tiny files, and every query pays to
open all of them.

In [ ]:
_base = spark.table(SOURCE_TABLE).limit(300000)

_base.limit(50000).write.format("delta").mode("overwrite").saveAsTable(BUSY)
for _i in range(5):
    (_base.filter(F.col("amount_inr") > _i)
          .limit(50000)
          .write.format("delta").mode("append").saveAsTable(BUSY))

display(
    spark.sql(f"DESCRIBE DETAIL {BUSY}")
         .select("format", "numFiles", "sizeInBytes")
)

### Compaction

In [ ]:
display(spark.sql(f"OPTIMIZE {BUSY}"))

In [ ]:
display(
    spark.sql(f"DESCRIBE DETAIL {BUSY}")
         .select("format", "numFiles", "sizeInBytes")
)

### Z-Order — arranging, not just compacting

`OPTIMIZE` makes files bigger. `ZORDER BY` decides **which rows sit together inside them**,
so that a query filtering on that column can skip whole files instead of opening them.

Choose columns you actually filter on. Z-ordering a column nobody queries costs the rewrite
and buys nothing.

In [ ]:
display(spark.sql(f"OPTIMIZE {BUSY} ZORDER BY (city)"))

### On this platform, some of that happens without you

Serverless applies predictive optimization — Databricks decides when to compact and how to
cluster, based on how the table is actually queried. The commands above still matter: they
are what you write on classic compute, on another vendor's Spark, and in every interview.

---
# 5 · What history costs

Every version you have travelled back to exists because its files were never deleted.
Storage is not free, and nothing has cleaned up after us.

`VACUUM` deletes files that are **no longer referenced** by the current version **and** older
than the retention period. The default retention is **7 days**.

In [ ]:
display(spark.sql(f"VACUUM {DIM} RETAIN 168 HOURS DRY RUN"))

### The trade you are making

`VACUUM` is the only operation in this course that **destroys the ability to time travel**.
Once the files are gone, `VERSION AS OF` returns an error rather than an answer — which is
the correct failure, but it is a failure.

The retention default exists to stop you deleting files a long-running query is still
reading. Lowering it is possible, and is how people corrupt production tables.

We run `DRY RUN` only.

---
# 6 · Change Data Feed

Time travel answers "what did the table look like then". Change Data Feed answers a
different question: **"what changed between then and now, row by row"** — which is what a
downstream system needs in order to update itself without re-reading everything.

In [ ]:
spark.sql(f"ALTER TABLE {DIM} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

_v = spark.sql(f"DESCRIBE HISTORY {DIM}").agg(F.max("version")).collect()[0][0]

spark.sql(f"UPDATE {DIM} SET tier = 'ELITE' WHERE customer_id = 105")
spark.sql(f"DELETE FROM {DIM} WHERE customer_id = 110")

print(f"reading changes made after version {_v}")

In [ ]:
display(spark.sql(f"""
    SELECT customer_id, name, city, tier, _change_type, _commit_version
    FROM table_changes('{DIM}', {_v} + 1)
    ORDER BY _commit_version, customer_id
"""))

### Four kinds of change

`insert`, `delete`, and — for an update — **two** rows: `update_preimage` and
`update_postimage`. Before and after, so a downstream consumer can reverse the old value as
well as apply the new one.

It has to be switched on before the changes happen. It cannot be turned on retrospectively
for history that has already been written.

---
## Your turn — 20 minutes

Build your own dimension. No dataset needed — type the rows.

**1 — A dimension of your own.**
Create a Delta table `products` with six rows: `product_id`, `name`, `category`, `price`.

**2 — Change one thing.**
`UPDATE` one price and `DELETE` one product. Read `DESCRIBE HISTORY` and find how many files
were added and removed for a single-row update.

**3 — A mixed file.**
Build a batch of four rows: two that already exist with changed prices, two that are new.
`MERGE` it in one statement. Report `numTargetRowsUpdated` and `numTargetRowsInserted`.

**4 — Keep the past.**
Build `products_history` with `valid_from`, `valid_to`, `is_current`. Apply a price change
using the staged two-row merge, and show both rows for that product.

**5 — Tidy up.**
Run `OPTIMIZE` on your table and compare `numFiles` before and after. Then run
`VACUUM ... DRY RUN` and read what it would remove.

**6 — Stretch.**
Turn on Change Data Feed, make one update, and write a query returning only the rows whose
price went **down**.

---
# Where this leaves you

You can now create, read, change and maintain a table. Everything so far has assumed the
data was already inside Databricks.

It never is. It arrives — as files, continuously, from systems you do not control, and
nobody tells you when.

That is Sunday: ingestion, and then streaming.

## Reset

In [ ]:
for _t in (DIM, SCD, BUSY):
    spark.sql(f"DROP TABLE IF EXISTS {_t}")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

print("session objects removed")